# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [ ]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu datasketch groq openai tqdm networkx spacy datasets langchain-community llama-index

In [ ]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
os.environ.setdefault("OMP_NUM_THREADS", "1")  # tránh OpenMP conflict giữa Torch và FAISS
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

load_dotenv()  # local; trên Colab secrets vẫn được ưu tiên bởi get_secret()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")
GROQ_FALLBACK_MODEL = get_secret("GROQ_FALLBACK_MODEL", "openai/gpt-oss-20b")
PIPELINE_PROVIDER = get_secret("PIPELINE_PROVIDER", "groq").lower()
PIPELINE_MODEL = get_secret("PIPELINE_MODEL", GROQ_MODEL)

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = PROJECT_ROOT / "reports"
for directory in [DATA_DIR, OUTPUT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
DATA_PATH = str(PROJECT_ROOT / "hackernoon_subset.csv")  # chạy được cả Colab và local
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` trỏ tới `hackernoon_subset.csv` trong thư mục làm việc hiện tại (`/content` trên Colab hoặc thư mục project khi chạy local).

In [ ]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

In [ ]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

# connect_neo4j()
# setup_graph_schema()

In [ ]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(x)).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

# raw_df = load_news(DATA_PATH)
# news_df = standardize_news(raw_df)
# chunks_df = build_chunks(news_df)
# display(chunks_df.head())

### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [ ]:
#@title 1.5b — Near-duplicate detection bằng MinHash/LSH (không O(N²))
from datasketch import MinHash, MinHashLSH

NEAR_DEDUP_LSH_THRESHOLD = 0.75   # candidate generation: ưu tiên recall
NEAR_DEDUP_VERIFY_THRESHOLD = 0.82  # hard merge threshold: exact 5-shingle Jaccard
NEAR_DEDUP_MIN_LENGTH_RATIO = 0.80
NEAR_DEDUP_SHINGLE_SIZE = 5
NEAR_DEDUP_NUM_PERM = 128

def _dedup_tokens(text):
    text = unicodedata.normalize("NFKC", norm_space(text)).lower()
    return re.findall(r"\w+", text, flags=re.UNICODE)

def _word_shingles(text, size=NEAR_DEDUP_SHINGLE_SIZE):
    tokens = _dedup_tokens(text)
    if len(tokens) < size:
        return frozenset([" ".join(tokens)]) if tokens else frozenset()
    return frozenset(" ".join(tokens[i:i + size]) for i in range(len(tokens) - size + 1))

def _exact_jaccard(left, right):
    union_size = len(left | right)
    return len(left & right) / union_size if union_size else 0.0

def _make_minhash(shingles, num_perm=NEAR_DEDUP_NUM_PERM):
    signature = MinHash(num_perm=num_perm, seed=SEED)
    signature.update_batch([s.encode("utf-8") for s in shingles])
    return signature

def near_deduplicate_minhash_lsh(
    df,
    lsh_threshold=NEAR_DEDUP_LSH_THRESHOLD,
    verify_threshold=NEAR_DEDUP_VERIFY_THRESHOLD,
    min_length_ratio=NEAR_DEDUP_MIN_LENGTH_RATIO,
    shingle_size=NEAR_DEDUP_SHINGLE_SIZE,
    num_perm=NEAR_DEDUP_NUM_PERM,
    min_shingles=20,
):
    """LSH sinh candidate; chỉ candidate mới được kiểm tra Jaccard chính xác.

    Canonical được xét theo bài dài nhất trước. Một bài chỉ merge khi nó vượt
    ngưỡng trực tiếp với canonical, tránh lỗi chain A~B, B~C nhưng A không ~C.
    Trả về (deduped_df, audit_df); mỗi bài bị bỏ có đúng một audit record.
    """
    if df.empty:
        return df.copy(), pd.DataFrame()
    if not 0 < lsh_threshold <= verify_threshold <= 1:
        raise ValueError("Cần 0 < lsh_threshold <= verify_threshold <= 1.")

    work = df.reset_index(drop=True).copy()
    shingles = [
        _word_shingles(f"{row.title} {row.text}", shingle_size)
        for row in work.itertuples(index=False)
    ]
    signatures = [_make_minhash(s, num_perm) for s in tqdm(shingles, desc="MinHash")]
    order = sorted(range(len(work)), key=lambda i: (-len(shingles[i]), i))

    lsh = MinHashLSH(threshold=lsh_threshold, num_perm=num_perm)
    canonical_of = {}
    audit_rows = []
    lsh_candidates = 0
    verified_pairs = 0

    for pos in tqdm(order, desc="LSH near-dedup"):
        key = str(pos)
        candidate_positions = [int(x) for x in lsh.query(signatures[pos])]
        lsh_candidates += len(candidate_positions)
        candidate_canonicals = sorted({canonical_of[x] for x in candidate_positions})

        best = None
        if len(shingles[pos]) >= min_shingles:
            for canonical_pos in candidate_canonicals:
                if len(shingles[canonical_pos]) < min_shingles:
                    continue
                verified_pairs += 1
                length_ratio = min(len(shingles[pos]), len(shingles[canonical_pos])) / max(
                    len(shingles[pos]), len(shingles[canonical_pos])
                )
                if length_ratio < min_length_ratio:
                    continue
                jaccard = _exact_jaccard(shingles[pos], shingles[canonical_pos])
                if jaccard >= verify_threshold and (best is None or jaccard > best[0]):
                    best = (jaccard, length_ratio, canonical_pos)

        if best is None:
            canonical_of[pos] = pos
        else:
            jaccard, length_ratio, canonical_pos = best
            canonical_of[pos] = canonical_pos
            duplicate = work.iloc[pos]
            canonical = work.iloc[canonical_pos]
            audit_rows.append({
                "duplicate_article_id": duplicate.article_id,
                "canonical_article_id": canonical.article_id,
                "jaccard_5gram": round(jaccard, 6),
                "length_ratio": round(length_ratio, 6),
                "duplicate_title": duplicate.title,
                "canonical_title": canonical.title,
                "duplicate_date": duplicate.published_date,
                "canonical_date": canonical.published_date,
                "duplicate_preview": duplicate.text[:300],
                "canonical_preview": canonical.text[:300],
                "decision": "merge",
                "review_label": "",  # điền TP hoặc FP khi audit thủ công
                "review_note": "",
            })

        # Index cả duplicate để tăng recall; khi merge vẫn verify trực tiếp với canonical.
        lsh.insert(key, signatures[pos])

    kept_positions = sorted(pos for pos, canonical_pos in canonical_of.items() if pos == canonical_pos)
    deduped = work.iloc[kept_positions].reset_index(drop=True)
    audit_columns = [
        "duplicate_article_id", "canonical_article_id", "jaccard_5gram",
        "length_ratio", "duplicate_title", "canonical_title",
        "duplicate_date", "canonical_date", "duplicate_preview",
        "canonical_preview", "decision", "review_label", "review_note",
    ]
    audit_df = pd.DataFrame(audit_rows, columns=audit_columns)
    print(
        f"Near dedup: {len(work):,} -> {len(deduped):,}; "
        f"LSH candidates={lsh_candidates:,}; exact checks={verified_pairs:,}; "
        f"full-pairwise baseline={len(work) * (len(work) - 1) // 2:,}"
    )
    return deduped, audit_df

def build_merge_audit_sample(audit_df, boundary_size=30, random_size=20):
    """Ưu tiên cặp sát threshold, cộng mẫu ngẫu nhiên để audit false positive."""
    if audit_df.empty:
        return audit_df.copy()
    boundary = audit_df.nsmallest(min(boundary_size, len(audit_df)), "jaccard_5gram")
    remaining = audit_df.drop(boundary.index)
    random_part = remaining.sample(min(random_size, len(remaining)), random_state=SEED)
    return pd.concat([boundary, random_part]).drop_duplicates().reset_index(drop=True)

def audit_false_positive_rate(audit_sample):
    labels = audit_sample["review_label"].astype(str).str.upper().str.strip()
    reviewed = labels.isin(["TP", "FP"])
    if not reviewed.any():
        return {"reviewed_pairs": 0, "false_positives": 0, "false_positive_rate": None}
    false_positives = int((labels[reviewed] == "FP").sum())
    return {
        "reviewed_pairs": int(reviewed.sum()),
        "false_positives": false_positives,
        "false_positive_rate": false_positives / int(reviewed.sum()),
    }

# Chạy sau cell loader:
# raw_df = load_news(DATA_PATH)
# news_exact_df = standardize_news(raw_df)
# news_df, near_dedup_audit = near_deduplicate_minhash_lsh(news_exact_df)
# near_dedup_audit.to_csv("near_dedup_audit_all.csv", index=False)
# merge_audit_sample = build_merge_audit_sample(near_dedup_audit)
# merge_audit_sample.to_csv("near_dedup_audit_sample.csv", index=False)
# chunks_df = build_chunks(news_df)
# display(merge_audit_sample.head())

#### Báo cáo thiết kế Near Dedup

- **Candidate generation:** MinHash `128` permutations trên word 5-shingles, tra bằng LSH ở threshold `0.75`. Đây là tầng recall; nó không tự quyết định merge.
- **Threshold merge:** chỉ merge khi Jaccard chính xác của hai tập 5-shingle `>= 0.82` và tỷ lệ độ dài `>= 0.80`. Bài dài nhất được xét làm canonical trước. Mỗi duplicate phải đạt ngưỡng trực tiếp với canonical, nên không có merge bắc cầu A~B~C khi A không gần C.
- **Độ phức tạp:** tạo chữ ký xấp xỉ `O(N × num_perm)` và chỉ verify `C` candidate do LSH sinh ra; không tính cosine/Jaccard trên toàn bộ `N(N-1)/2` cặp. Số `LSH candidates` và `exact checks` được in khi chạy để chứng minh.
- **False positive có thể xảy ra:** bài dùng chung boilerplate/template, press release về cùng sự kiện nhưng có nội dung biên tập khác, hoặc bài ngắn có ít shingle. Mitigation gồm 5-shingle, `min_shingles=20`, length ratio và hard Jaccard. Không báo một tỷ lệ FP giả định: tỷ lệ thực nghiệm được tính bằng `audit_false_positive_rate()` sau khi reviewer gắn nhãn `TP`/`FP`.
- **Audit cặp merge:** `near_dedup_audit_all.csv` lưu một dòng cho mọi bài bị loại, gồm ID canonical/duplicate, score, length ratio, title, ngày và preview. `near_dedup_audit_sample.csv` ưu tiên 30 cặp sát threshold (rủi ro cao) cộng 20 cặp ngẫu nhiên. Reviewer đọc nội dung gốc theo hai ID, điền `review_label=TP|FP` và ghi lý do vào `review_note`. Nếu FP cao, tăng hard threshold hoặc bổ sung rule theo domain/date trước khi chạy lại.
- **Trade-off:** MinHash phù hợp repost/chỉnh sửa nhẹ dựa trên lexical overlap; paraphrase sâu có thể là false negative. Khi cần bắt paraphrase, có thể thay candidate generation bằng embedding+FAISS ANN nhưng vẫn phải giữ bước verify/audit tương tự.

In [ ]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    active_model = model or GROQ_MODEL
    if not active_model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last, fallback_used = None, False
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": active_model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                    "model": active_model,
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            message = str(e).lower()
            if (
                not fallback_used
                and active_model != GROQ_FALLBACK_MODEL
                and ("model_not_found" in message or "does not exist" in message)
            ):
                active_model = GROQ_FALLBACK_MODEL
                fallback_used = True
                print(f"Groq model unavailable; fallback -> {active_model}")
                continue
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

pipeline_openai_client = None

def pipeline_chat(messages, model=None, json_mode=False):
    """Provider-neutral chat for coref/extraction/seed/generation."""
    if PIPELINE_PROVIDER == "groq":
        return groq_chat(
            messages, model=model or PIPELINE_MODEL or GROQ_MODEL, json_mode=json_mode
        )
    if PIPELINE_PROVIDER != "openai":
        raise ValueError("PIPELINE_PROVIDER must be groq or openai.")
    if not OPENAI_API_KEY:
        raise RuntimeError("Thiếu OPENAI_API_KEY cho pipeline provider=openai.")

    global pipeline_openai_client
    if pipeline_openai_client is None:
        from openai import OpenAI
        pipeline_openai_client = OpenAI(api_key=OPENAI_API_KEY)
    active_model = model or PIPELINE_MODEL or JUDGE_MODEL
    kwargs = {
        "model": active_model, "messages": messages, "temperature": 0.0,
    }
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    response = pipeline_openai_client.chat.completions.create(**kwargs)
    usage = {"model": active_model}
    if response.usage:
        usage.update({
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        })
    return response.choices[0].message.content, usage

def pipeline_json(system, user, model=None):
    text, usage = pipeline_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model, json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [ ]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = pipeline_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

# extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
# coref_df = run_coref(extraction_source)
# extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph contract

Mỗi node có label gốc `Entity` và **đúng một** subtype: `Company`, `Person` hoặc `Technology`. `Entity.id` là khóa ổn định tạo từ `(entity_type, normalized_name)`; alias chỉ là thuộc tính, không tạo node mới.

| Relation | Source hợp lệ | Target hợp lệ | Ví dụ hướng đúng |
|---|---|---|---|
| `ACQUIRED` | Company | Company | Google → Fitbit |
| `DEVELOPED` | Company, Person | Technology | OpenAI → GPT-4 |
| `INVESTED_IN` | Company, Person | Company | Microsoft → OpenAI |
| `FOUNDED` | Person | Company | Steve Jobs → Apple |
| `WORKED_AT` | Person | Company | Satya Nadella → Microsoft |
| `PARTNERED_WITH` | Company | Company | Microsoft → OpenAI |
| `USES` | Company | Technology | GitHub → Git |
| `LEADS` | Person | Company | Tim Cook → Apple |

### Edge contract và provenance

- Khóa edge ổn định: SHA-1 của `(source_id, relation, target_id, source_chunk_id)`. Cùng một fact ở hai chunk là hai evidence edge khác nhau; chạy lại cùng chunk là idempotent.
- Bắt buộc: `source_chunk_id` không rỗng và `published_date` hợp lệ theo `YYYY-MM-DD`.
- `evidence` phải là trích đoạn nguyên văn trong chunk; `confidence` nằm trong `[0, 1]`. Mặc định chỉ nhận `confidence >= 0.60`.
- Relation, source type, target type và hướng phải qua allowlist/type contract trước khi ghép label/type động vào Cypher. Không lấy trực tiếp relation do LLM trả về để nội suy câu query.
- Mọi candidate bị loại phải vào `triple_rejections_df` với `reason` và payload gốc để audit; không bỏ im lặng.

In [ ]:
#@title 2.1 — High-precision NER + RE extraction với validation audit
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
RELATION_TYPE_RULES = {
    "ACQUIRED": {("Company", "Company")},
    "DEVELOPED": {("Company", "Technology"), ("Person", "Technology")},
    "INVESTED_IN": {("Company", "Company"), ("Person", "Company")},
    "FOUNDED": {("Person", "Company")},
    "WORKED_AT": {("Person", "Company")},
    "PARTNERED_WITH": {("Company", "Company")},
    "USES": {("Company", "Technology")},
    "LEADS": {("Person", "Company")},
}
ALLOWED_RELATIONS = set(RELATION_TYPE_RULES)
MIN_EXTRACTION_CONFIDENCE = 0.60
GENERIC_ENTITY_MENTIONS = {
    "it", "they", "he", "she", "the company", "the startup",
    "the firm", "this company", "the organization",
}
SCHEMA_FOR_PROMPT = {
    rel: [f"{source_type}->{target_type}" for source_type, target_type in sorted(pairs)]
    for rel, pairs in RELATION_TYPE_RULES.items()
}

EXTRACT_SYSTEM = f"""
You are a conservative information-extraction component for a production knowledge graph.
Allowed directed relation schema: {json.dumps(SCHEMA_FOR_PROMPT, sort_keys=True)}
Extract only facts explicitly stated in the supplied text. Prefer precision over recall.
Copy source and target surface forms exactly from the evidence.
Evidence must be one short verbatim substring of the supplied text containing both entities.
Do not infer facts from general knowledge, headlines, or an ambiguous pronoun.
Never emit generic entities such as 'the company'; omit them if coreference is unresolved.
Do not convert planned, proposed, considered, or committed actions into completed relations.
For ACQUIRED, source is the buyer and target is the acquired company; omit asset-only transfers that do not fit this schema.
If direction, endpoint type, or evidence is uncertain, omit the relation.
Confidence must reflect evidence strength (e.g. 0.95 explicit, 0.75 clear); never copy a 0 placeholder.
Omit candidates below 0.60 confidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return exactly this shape:
{{
  "items": [
    {{
      "chunk_id": "copy input chunk_id exactly",
      "relations": [
        {{
          "source": "verbatim entity surface form",
          "source_type": "Company|Person|Technology",
          "relation": "allowed directed relation",
          "target": "verbatim entity surface form",
          "target_type": "Company|Person|Technology",
          "evidence": "verbatim substring containing source and target",
          "confidence": 0.95
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return pipeline_json(EXTRACT_SYSTEM, prompt)

def _safe_confidence(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    return value if np.isfinite(value) and 0.0 <= value <= 1.0 else None

def _match_text(value):
    return unicodedata.normalize("NFKC", norm_space(value)).casefold()

def _modality_rejection(relation, evidence):
    text = _match_text(evidence)
    if relation == "ACQUIRED" and any(phrase in text for phrase in [
        "to acquire", "will acquire", "plans to acquire", "planned acquisition",
        "agreement to acquire", "agreed to acquire", "to be transferred", "considering",
    ]):
        return "NON_COMPLETED_ACQUISITION"
    if relation in {"DEVELOPED", "USES"} and any(phrase in text for phrase in [
        "plans to", "planning to", "considering", "committed to",
        "could use", "may use", "will develop",
    ]):
        return "NON_COMPLETED_OR_TENTATIVE_ACTION"
    return None

def _validate_relation_candidate(candidate, chunk_id, chunk_meta, min_confidence):
    if not isinstance(candidate, dict):
        return None, ["CANDIDATE_NOT_OBJECT"]

    source = norm_space(candidate.get("source"))
    target = norm_space(candidate.get("target"))
    source_type = norm_space(candidate.get("source_type"))
    target_type = norm_space(candidate.get("target_type"))
    relation = norm_space(candidate.get("relation")).upper()
    evidence = norm_space(candidate.get("evidence"))
    confidence = _safe_confidence(candidate.get("confidence"))
    published_date = norm_space(chunk_meta.get("published_date"))
    chunk_text = norm_space(chunk_meta.get("text"))
    reasons = []

    if not source or not target:
        reasons.append("EMPTY_ENDPOINT")
    if source_type not in ALLOWED_NODE_TYPES or target_type not in ALLOWED_NODE_TYPES:
        reasons.append("INVALID_NODE_TYPE")
    if relation not in ALLOWED_RELATIONS:
        reasons.append("RELATION_NOT_ALLOWLISTED")
    elif (source_type, target_type) not in RELATION_TYPE_RULES[relation]:
        reasons.append("INVALID_RELATION_DIRECTION_OR_ENDPOINT_TYPES")
    if source and target and _match_text(source) == _match_text(target):
        reasons.append("SELF_RELATION")
    if _match_text(source) in GENERIC_ENTITY_MENTIONS or _match_text(target) in GENERIC_ENTITY_MENTIONS:
        reasons.append("GENERIC_UNRESOLVED_ENTITY")
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", published_date):
        reasons.append("MISSING_OR_INVALID_PUBLISHED_DATE")
    if not evidence:
        reasons.append("EMPTY_EVIDENCE")
    elif _match_text(evidence) not in _match_text(chunk_text):
        reasons.append("EVIDENCE_NOT_VERBATIM_IN_CHUNK")
    else:
        evidence_norm = _match_text(evidence)
        if source and _match_text(source) not in evidence_norm:
            reasons.append("SOURCE_NOT_IN_EVIDENCE")
        if target and _match_text(target) not in evidence_norm:
            reasons.append("TARGET_NOT_IN_EVIDENCE")
    if confidence is None:
        reasons.append("INVALID_CONFIDENCE")
    elif confidence < min_confidence:
        reasons.append("BELOW_CONFIDENCE_THRESHOLD")
    modality_reason = _modality_rejection(relation, evidence)
    if modality_reason:
        reasons.append(modality_reason)

    if reasons:
        return None, reasons
    return {
        "source_raw": source,
        "source_type": source_type,
        "relation": relation,
        "target_raw": target,
        "target_type": target_type,
        "source_chunk_id": chunk_id,
        "published_date": published_date,
        "evidence": evidence,
        "confidence": confidence,
    }, []

def run_extraction(source_df, batch_size=4, min_confidence=MIN_EXTRACTION_CONFIDENCE):
    meta = {}
    for row in source_df.itertuples(index=False):
        meta[row.chunk_id] = {
            "published_date": row.published_date,
            "text": getattr(row, "resolved_text", None) or row.text,
        }

    triples, errors, rejections = [], [], []
    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start + batch_size]
        responses = []
        try:
            obj, _ = extract_batch(batch)
            responses.append(obj)
        except Exception as batch_exc:
            # JSON-mode có thể fail trên batch dài; retry từng chunk để không mất cả batch.
            for row_pos in range(len(batch)):
                one = batch.iloc[row_pos:row_pos + 1]
                try:
                    obj, _ = extract_batch(one)
                    responses.append(obj)
                except Exception as exc:
                    errors.append({
                        "batch_start": start + row_pos,
                        "chunk_ids": one["chunk_id"].tolist(),
                        "error": f"batch={batch_exc}; single={exc}",
                    })

        for response_obj in responses:
            items = response_obj.get("items", []) if isinstance(response_obj, dict) else []
            for item in items:
                if not isinstance(item, dict):
                    rejections.append({
                        "source_chunk_id": "", "reason": "ITEM_NOT_OBJECT",
                        "candidate_json": json.dumps(item, ensure_ascii=False, default=str),
                    })
                    continue
                chunk_id = item.get("chunk_id")
                if chunk_id not in meta:
                    rejections.append({
                        "source_chunk_id": norm_space(chunk_id), "reason": "UNKNOWN_CHUNK_ID",
                        "candidate_json": json.dumps(item, ensure_ascii=False, default=str),
                    })
                    continue

                for candidate in item.get("relations", []):
                    triple, reasons = _validate_relation_candidate(
                        candidate, chunk_id, meta[chunk_id], min_confidence
                    )
                    if triple is not None:
                        triples.append(triple)
                    else:
                        rejections.append({
                            "source_chunk_id": chunk_id,
                            "reason": "|".join(reasons),
                            "candidate_json": json.dumps(candidate, ensure_ascii=False, default=str),
                        })

    triple_columns = [
        "source_raw", "source_type", "relation", "target_raw",
        "target_type", "source_chunk_id", "published_date", "evidence", "confidence",
    ]
    triples_df = pd.DataFrame(triples, columns=triple_columns).drop_duplicates(
        ["source_raw", "source_type", "relation", "target_raw", "target_type", "source_chunk_id"]
    ).reset_index(drop=True)
    errors_df = pd.DataFrame(errors, columns=["batch_start", "chunk_ids", "error"])
    rejections_df = pd.DataFrame(
        rejections, columns=["source_chunk_id", "reason", "candidate_json"]
    )
    print(
        f"Accepted triples={len(triples_df):,}; rejected candidates={len(rejections_df):,}; "
        f"failed batches={len(errors_df):,}"
    )
    return triples_df, errors_df, rejections_df

# raw_triples_df, extraction_errors_df, triple_rejections_df = run_extraction(extraction_source)
# display(raw_triples_df.head())
# display(triple_rejections_df["reason"].value_counts())

## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [ ]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def _looks_like_ticker(name):
    return bool(re.fullmatch(r"[A-Z]{1,5}(?:[.-][A-Z])?", norm_space(name)))

def _token_jaccard(a, b):
    left, right = set(norm_entity(a).split()), set(norm_entity(b).split())
    return len(left & right) / len(left | right) if left | right else 0.0

def merge_guard(a, b, entity_type):
    raw_a, raw_b = norm_space(a), norm_space(b)
    na, nb = norm_entity(raw_a), norm_entity(raw_b)
    if na == nb:
        return True, "EXACT_NORMALIZED"

    # Ticker chỉ merge qua MANUAL_ALIASES; embedding không đủ an toàn.
    if _looks_like_ticker(raw_a) or _looks_like_ticker(raw_b):
        return False, "TICKER_REQUIRES_MANUAL_ALIAS"

    if entity_type == "Company":
        sa, sb = strip_suffix(raw_a), strip_suffix(raw_b)
        if sa and sa == sb:
            return True, "CORPORATE_SUFFIX_ONLY"
        ratio = SequenceMatcher(None, sa, sb).ratio()
        ok = ratio >= 0.88 and _token_jaccard(sa, sb) >= 0.60
        return ok, "COMPANY_LEXICAL_MATCH" if ok else "COMPANY_NAME_DIVERGENCE"

    if entity_type == "Person":
        ta, tb = na.replace(".", "").split(), nb.replace(".", "").split()
        if len(ta) < 2 or len(tb) < 2 or ta[-1] != tb[-1]:
            return False, "PERSON_SURNAME_MISMATCH_OR_INCOMPLETE"
        first_compatible = ta[0] == tb[0] or ta[0][0] == tb[0][0] and (len(ta[0]) == 1 or len(tb[0]) == 1)
        middle_compatible = ta[1:-1] == tb[1:-1] or not ta[1:-1] or not tb[1:-1]
        ok = first_compatible and middle_compatible
        return ok, "PERSON_INITIAL_VARIANT" if ok else "PERSON_GIVEN_NAME_CONFLICT"

    # Technology/product: containment thường là product family khác nhau (Apple vs Apple Music).
    if na in nb or nb in na:
        return False, "PRODUCT_CONTAINMENT_CONFLICT"
    ratio = SequenceMatcher(None, na, nb).ratio()
    ok = ratio >= 0.90 and _token_jaccard(na, nb) >= 0.67
    return ok, "TECHNOLOGY_LEXICAL_MATCH" if ok else "TECHNOLOGY_NAME_DIVERGENCE"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if t == "Company" and norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL",
                "guard_reason": "MANUAL_ALIAS"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        if len(keys) == 1:
            mapping[keys[0]] = names[0]
            continue
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexHNSWFlat(vecs.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = 80
        index.hnsw.efSearch = 64
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j:
                    continue
                if float(score) < threshold:
                    audit.append({
                        "type": typ, "left": names[i], "right": names[j],
                        "similarity": float(score),
                        "decision": "REJECT_BELOW_THRESHOLD",
                        "guard_reason": f"COSINE_LT_{threshold:.2f}",
                    })
                    continue
                ok, guard_reason = merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD",
                    "guard_reason": guard_reason,
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        manual = MANUAL_ALIASES.get(n) if typ == "Company" else None
        return mapping.get((typ, n), manual or name)

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

# entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
# triples_df = canonicalize_triples(raw_triples_df, entity_map)
# display(entity_resolution_audit_df.head(20))

In [ ]:
#@title 2.3 — Preflight validation + idempotent UNWIND bulk insert
EDGE_REQUIRED_COLUMNS = {
    "source_id", "source_name", "source_name_norm", "source_type",
    "target_id", "target_name", "target_name_norm", "target_type",
    "relation", "source_chunk_id", "published_date", "evidence", "confidence",
}

def validate_triples_for_insert(triples_df, min_confidence=MIN_EXTRACTION_CONFIDENCE):
    missing = EDGE_REQUIRED_COLUMNS - set(triples_df.columns)
    if missing:
        raise ValueError(f"Missing triple columns: {sorted(missing)}")

    accepted, rejected = [], []
    for row_index, row in triples_df.iterrows():
        record = row.to_dict()
        relation = norm_space(record.get("relation")).upper()
        source_type = norm_space(record.get("source_type"))
        target_type = norm_space(record.get("target_type"))
        source_id = norm_space(record.get("source_id"))
        target_id = norm_space(record.get("target_id"))
        chunk_id = norm_space(record.get("source_chunk_id"))
        published_date = norm_space(record.get("published_date"))
        evidence = norm_space(record.get("evidence"))
        source_name = norm_space(record.get("source_name") or record.get("source_raw"))
        target_name = norm_space(record.get("target_name") or record.get("target_raw"))
        confidence = _safe_confidence(record.get("confidence"))
        reasons = []

        if not source_id or not target_id:
            reasons.append("EMPTY_NODE_ID")
        if source_id and source_id == target_id:
            reasons.append("SELF_RELATION")
        if _match_text(source_name) in GENERIC_ENTITY_MENTIONS or _match_text(target_name) in GENERIC_ENTITY_MENTIONS:
            reasons.append("GENERIC_UNRESOLVED_ENTITY")
        if source_type not in ALLOWED_NODE_TYPES or target_type not in ALLOWED_NODE_TYPES:
            reasons.append("INVALID_NODE_TYPE")
        if relation not in ALLOWED_RELATIONS:
            reasons.append("RELATION_NOT_ALLOWLISTED")
        elif (source_type, target_type) not in RELATION_TYPE_RULES[relation]:
            reasons.append("INVALID_RELATION_DIRECTION_OR_ENDPOINT_TYPES")
        if not chunk_id:
            reasons.append("EMPTY_SOURCE_CHUNK_ID")
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", published_date):
            reasons.append("MISSING_OR_INVALID_PUBLISHED_DATE")
        if not evidence:
            reasons.append("EMPTY_EVIDENCE")
        if confidence is None:
            reasons.append("INVALID_CONFIDENCE")
        elif confidence < min_confidence:
            reasons.append("BELOW_CONFIDENCE_THRESHOLD")
        modality_reason = _modality_rejection(relation, evidence)
        if modality_reason:
            reasons.append(modality_reason)

        if reasons:
            rejected.append({
                "row_index": row_index,
                "source_id": source_id,
                "relation": relation,
                "target_id": target_id,
                "source_chunk_id": chunk_id,
                "reason": "|".join(reasons),
            })
            continue

        record.update({
            "relation": relation,
            "source_type": source_type,
            "target_type": target_type,
            "source_id": source_id,
            "target_id": target_id,
            "source_chunk_id": chunk_id,
            "published_date": published_date,
            "evidence": evidence,
            "confidence": confidence,
            "edge_id": sha1(f"{source_id}|{relation}|{target_id}|{chunk_id}"),
        })
        accepted.append(record)

    accepted_columns = list(triples_df.columns) + (
        [] if "edge_id" in triples_df.columns else ["edge_id"]
    )
    accepted_df = pd.DataFrame(accepted, columns=accepted_columns).drop_duplicates("edge_id")
    rejected_df = pd.DataFrame(
        rejected,
        columns=["row_index", "source_id", "relation", "target_id", "source_chunk_id", "reason"],
    )
    return accepted_df.reset_index(drop=True), rejected_df

def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    if nodes_df.empty:
        return 0
    invalid_types = set(nodes_df["type"]) - ALLOWED_NODE_TYPES
    if invalid_types:
        raise ValueError(f"Node types are not allowlisted: {sorted(invalid_types)}")

    written = 0
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        ON CREATE SET n.created_at=datetime()
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=reduce(acc=coalesce(n.aliases, []), x IN row.aliases |
                CASE WHEN x IN acc THEN acc ELSE acc + x END),
            n.aliases_norm=reduce(acc=coalesce(n.aliases_norm, []), x IN row.aliases_norm |
                CASE WHEN x IN acc THEN acc ELSE acc + x END),
            n.updated_at=datetime()
        RETURN count(n) AS written
        """
        for b in batches(part.to_dict("records"), batch_size):
            result = run_cypher(query, rows=b)
            written += result[0]["written"] if result else 0
    return written

def bulk_insert_edges(triples_df, batch_size=1000, strict=True):
    valid_df, rejected_df = validate_triples_for_insert(triples_df)
    if strict and not rejected_df.empty:
        counts = rejected_df["reason"].value_counts().to_dict()
        raise ValueError(f"Refusing bulk insert: {len(rejected_df)} invalid triples: {counts}")

    written = 0
    for rel in sorted(ALLOWED_RELATIONS):
        part = valid_df[valid_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{id: row.edge_id}}]->(t)
        ON CREATE SET r.created_at=datetime(),
                      r.source_chunk_id=row.source_chunk_id,
                      r.published_date=row.published_date,
                      r.evidence=row.evidence,
                      r.confidence=row.confidence
        ON MATCH SET r.evidence=CASE
                         WHEN row.confidence >= coalesce(r.confidence, -1.0) THEN row.evidence
                         ELSE r.evidence END,
                     r.confidence=CASE
                         WHEN row.confidence >= coalesce(r.confidence, -1.0) THEN row.confidence
                         ELSE r.confidence END,
                     r.updated_at=datetime()
        RETURN count(r) AS written
        """

        cols = [
            "edge_id", "source_id", "target_id", "source_chunk_id",
            "published_date", "evidence", "confidence",
        ]
        for b in batches(part[cols].to_dict("records"), batch_size):
            result = run_cypher(query, rows=b)
            written += result[0]["written"] if result else 0

    return {
        "accepted_edges": len(valid_df),
        "rejected_edges": len(rejected_df),
        "matched_and_written": written,
    }, rejected_df

# triples_df, insertion_rejections_df = validate_triples_for_insert(triples_df)
# assert insertion_rejections_df.empty, insertion_rejections_df["reason"].value_counts().to_dict()
# nodes_df = build_nodes(triples_df)
# inserted_nodes = bulk_insert_nodes(nodes_df)
# edge_insert_stats, edge_insert_rejections_df = bulk_insert_edges(triples_df)
# print({"inserted_nodes": inserted_nodes, **edge_insert_stats})

In [ ]:
#@title 2.4 — Graph contract sanity checks
def graph_checks():
    invalid_provenance = run_cypher("""
    MATCH (:Entity)-[r]->(:Entity)
    WHERE r.source_chunk_id IS NULL OR trim(toString(r.source_chunk_id)) = ''
       OR r.published_date IS NULL OR trim(toString(r.published_date)) = ''
    RETURN count(r) AS n
    """)[0]["n"]

    invalid_relation_types = run_cypher("""
    MATCH (:Entity)-[r]->(:Entity)
    WHERE NOT type(r) IN $allowed_relations
    RETURN count(r) AS n
    """, allowed_relations=sorted(ALLOWED_RELATIONS))[0]["n"]

    invalid_endpoint_types = run_cypher("""
    MATCH (s:Entity)-[r]->(t:Entity)
    WHERE CASE type(r)
      WHEN 'ACQUIRED' THEN NOT (s:Company AND t:Company)
      WHEN 'DEVELOPED' THEN NOT ((s:Company OR s:Person) AND t:Technology)
      WHEN 'INVESTED_IN' THEN NOT ((s:Company OR s:Person) AND t:Company)
      WHEN 'FOUNDED' THEN NOT (s:Person AND t:Company)
      WHEN 'WORKED_AT' THEN NOT (s:Person AND t:Company)
      WHEN 'PARTNERED_WITH' THEN NOT (s:Company AND t:Company)
      WHEN 'USES' THEN NOT (s:Company AND t:Technology)
      WHEN 'LEADS' THEN NOT (s:Person AND t:Company)
      ELSE true
    END
    RETURN count(r) AS n
    """)[0]["n"]

    invalid_node_subtypes = run_cypher("""
    MATCH (n:Entity)
    WITH n, [label IN labels(n) WHERE label IN $node_types] AS subtypes
    WHERE size(subtypes) <> 1
    RETURN count(n) AS n
    """, node_types=sorted(ALLOWED_NODE_TYPES))[0]["n"]

    invalid_edge_ids = run_cypher("""
    MATCH (:Entity)-[r]->(:Entity)
    WITH r.id AS id, count(r) AS occurrences
    WHERE id IS NULL OR occurrences > 1
    RETURN coalesce(sum(CASE WHEN id IS NULL THEN occurrences ELSE occurrences - 1 END), 0) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH (:Entity)-[r]->(:Entity) RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid_provenance,
        "invalid_relation_types": invalid_relation_types,
        "invalid_endpoint_types": invalid_endpoint_types,
        "invalid_node_subtypes": invalid_node_subtypes,
        "missing_or_duplicate_edge_ids": invalid_edge_ids,
    }
    print(counts)
    assert all(counts[key] == 0 for key in [
        "invalid_provenance_edges", "invalid_relation_types",
        "invalid_endpoint_types", "invalid_node_subtypes",
        "missing_or_duplicate_edge_ids",
    ])

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

# graph_counts, top_degree_df = graph_checks()

# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [ ]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

# build_flat_index(chunks_df)

## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [ ]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = pipeline_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = np.einsum("ij,j->i", entity_match_vectors[idxs], qv, optimize=True)
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

# build_entity_matcher(nodes_df)

In [ ]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [ ]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = pipeline_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=PIPELINE_MODEL
    )
    return {
        "answer": text.strip(),
        "generation_latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
        "model": usage.get("model", GROQ_MODEL),
    }

def answer_flat_rag(question):
    t0 = time.perf_counter()
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out["latency_s"] = time.perf_counter() - t0
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    t0 = time.perf_counter()
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out["latency_s"] = time.perf_counter() - t0
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [ ]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = DATA_DIR / "graphrag_golden_50_first5000.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

In [ ]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [ ]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = OUTPUT_DIR / "graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

# validate_golden(golden_df, require_answers=True)
# eval_results_df = run_evaluation(golden_df)
# display(eval_results_df)

In [ ]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

# comparison_df = comparison_table(eval_results_df)
# display(comparison_df)
# eval_results_df.to_csv(OUTPUT_DIR / "graphrag_eval_results.csv", index=False)
# comparison_df.to_csv(OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv", index=False)

# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [ ]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

# test_supernode_policy()
# show_resolution_audit(entity_resolution_audit_df)

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = pipeline_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# Kết quả integration run — 19/08/2026

> Lần chạy đo được thực hiện bằng `scripts/run_lab_test.py` trên local; các CSV chi tiết nằm trong `outputs/`. Output lỗi Colab cũ đã được xóa để notebook không chứa trạng thái chạy lỗi.

- **Dữ liệu:** đọc 5.000 dòng đầu từ `hackernoon_subset.csv`; sau lọc/exact dedup còn 2.105 bài, MinHash/LSH near-dedup còn 2.096 bài (merge 9). Corpus benchmark kiểm soát gồm 100 bài và 100 chunks.
- **Near-dedup:** MinHash 128 permutations, word 5-shingles; LSH candidate threshold `0.75`, exact Jaccard merge threshold `0.82`, length ratio `>= 0.80`. Audit thủ công 9/9 cặp merge là true positive, false-positive quan sát `0/9` (mẫu nhỏ).
- **Extraction:** 15 triples ban đầu; failure audit loại 2 triples do modality/generic-entity guard, còn 13 triples.
- **Neo4j cuối cùng:** 20 nodes, 13 edges; 0 edge thiếu provenance, 0 relation/type sai allowlist, 0 endpoint sai schema, 0 edge ID thiếu/trùng.
- **Entity Resolution:** FAISS HNSW ANN, cosine threshold `0.90`, lexical/type guards; audit 43 candidates.
- **Benchmark 15 câu:** Flat/Graph comprehensiveness `3.733/3.800`; faithfulness `3.867/3.867`; multi-hop `3.733/3.800`; latency `1.949s/5.593s`; token `600.7/540.8`; evidence recall cùng `0.811`.
- **Kiểm thử:** `pytest` đạt 5/5; super-node synthetic test xác nhận node degree 150 chỉ lấy 50 cạnh và toàn traversal không vượt 250 cạnh.
- **Lưu ý:** benchmark là snapshot trước khi audit loại 2 cạnh sai; graph trên Aura và `validated_triples.csv` đã phản ánh bản sửa sau audit.

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau